# Forecast EV y modelo energetico
Baseline inicial para ETAPA 1 y calculo parametrico de ETAPA 2.

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

DATABASE_URL = os.getenv('DATABASE_URL', 'mysql+pymysql://root@127.0.0.1/proyecto_ev_colombia')engine = create_engine(DATABASE_URL)

In [ ]:
df = pd.read_sql('SELECT * FROM vehiculos_ev', engine)
df.columns = [column.lower() for column in df.columns]
df.head()

In [ ]:
feature_columns = ['anio', 'departamento', 'tipo_vehiculo']
target_column = 'cantidad_ev'
model_df = df[feature_columns + [target_column]].dropna().copy()
train_df = model_df.sort_values('anio').iloc[:-max(1, len(model_df) // 5)]
test_df = model_df.sort_values('anio').iloc[-max(1, len(model_df) // 5):]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), ['anio']),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore'))
        ]), ['departamento', 'tipo_vehiculo'])
    ]
)
model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42, n_estimators=200))
])
model.fit(train_df[feature_columns], train_df[target_column])
predictions = model.predict(test_df[feature_columns])
mean_absolute_error(test_df[target_column], predictions)

In [ ]:
pred_df = test_df.copy()
pred_df['cantidad_ev_pred'] = predictions
pred_df['kwh_promedio'] = df['kwh_promedio'].fillna(df['kwh_promedio'].median()) if 'kwh_promedio' in df.columns else 0
pred_df['potencia_carga'] = df['potencia_carga'].fillna(df['potencia_carga'].median()) if 'potencia_carga' in df.columns else 0
pred_df['simultaneidad'] = df['simultaneidad'].fillna(0.3) if 'simultaneidad' in df.columns else 0.3
pred_df['consumo_energetico'] = pred_df['cantidad_ev_pred'] * pred_df['kwh_promedio']
pred_df['demanda_futura'] = pred_df['cantidad_ev_pred'] * pred_df['potencia_carga'] * pred_df['simultaneidad']
pred_df.head()